<a href="https://colab.research.google.com/github/gopaps/MachineLearning/blob/main/TUGAS%20PERBAIKAN/Chapter_4_Perbaikan_Anda_Figo__Haq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4
membahas pentingnya representasi data dan rekayasa fitur (feature engineering) dalam machine learning. Bab ini menggarisbawahi bagaimana kita dapat memanfaatkan variabel kategori menggunakan metode seperti One-Hot Encoding agar sesuai dengan algoritma machine learning. Proses ini bertujuan untuk mengonversi data kategori menjadi bentuk numerik tanpa memperkenalkan hubungan ordinal palsu antar kategori

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import mean_squared_error, accuracy_score
import torch


In [2]:
# Periksa apakah GPU tersedia
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU tersedia. Menggunakan:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU tidak tersedia. Menggunakan CPU.")

GPU tersedia. Menggunakan: Tesla T4


In [3]:
# Membuat dataset simulasi untuk klasifikasi
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=3, random_state=42)

# Konversi data menjadi tensor torch untuk potensi penggunaan GPU
X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y, dtype=torch.long, device=device)

# Membagi data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [4]:
# One-Hot Encoding (contoh pada data kategorikal simulasi)
categorical_data = np.random.choice(['A', 'B', 'C'], size=1000)
one_hot_encoder = OneHotEncoder()
categorical_encoded = one_hot_encoder.fit_transform(categorical_data.reshape(-1, 1)).toarray()
print("Contoh hasil One-Hot Encoding:")
print(categorical_encoded[:5])

# Diskritisasi atau Binning data numerik
scaler = StandardScaler()
binned_data = np.digitize(scaler.fit_transform(X[:, 0].reshape(-1, 1)), bins=[-1, 0, 1])
print("Contoh hasil Binning:")
print(binned_data[:10])

Contoh hasil One-Hot Encoding:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
Contoh hasil Binning:
[[2]
 [0]
 [3]
 [0]
 [1]
 [3]
 [1]
 [0]
 [2]
 [0]]


In [5]:
# Interaksi dan Polinomial
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
print("Dimensi data setelah transformasi polinomial:", X_poly.shape)

# Seleksi Fitur otomatis
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X_train, y_train)
print("Fitur yang dipilih:", selector.get_support(indices=True))

Dimensi data setelah transformasi polinomial: (1000, 65)
Fitur yang dipilih: [1 4 5 8 9]


In [6]:
# Model sederhana dengan fitur terpilih
model = DecisionTreeClassifier(random_state=42)
model.fit(X_selected, y_train)
X_test_selected = selector.transform(X_test)
y_pred = model.predict(X_test_selected)
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi model dengan fitur terpilih: {accuracy * 100:.2f}%")


Akurasi model dengan fitur terpilih: 89.67%


In [7]:
# Pipeline untuk preprocessing dan modeling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('selector', SelectKBest(score_func=f_classif, k=5)),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred_pipeline = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred_pipeline)
print(f"Mean Squared Error dari Pipeline: {mse:.2f}")


Mean Squared Error dari Pipeline: 0.16


Bab ini juga mengeksplorasi cara-cara untuk merekayasa fitur baru, seperti melalui binning (diskritisasi), interaksi antar fitur, dan transformasi polinomial. Teknik ini sangat berguna untuk model linier yang biasanya membutuhkan fitur tambahan agar dapat menangkap pola kompleks. Sebaliknya, model nonlinier seperti random forests dan SVMs mampu menangani tugas kompleks tanpa eksplisit memperluas ruang fitur.

Lebih jauh, bab ini membahas pentingnya seleksi fitur otomatis, seperti dengan statistik univariat, metode berbasis model, atau seleksi iteratif, untuk meningkatkan efisiensi dan kinerja model. Dengan memanfaatkan pengetahuan ahli dalam mendesain fitur, kita dapat menciptakan model yang lebih relevan terhadap domain spesifik. Inti dari bab ini adalah bahwa pemilihan dan rekayasa fitur yang tepat sering kali menjadi komponen paling penting dalam membangun solusi machine learning yang berhasil.